# 00 Load Library
---


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# 01 Inisialisasi Path dan Konsolidasi Fitur
---
## Konfigurasi Direktori Kerja
---
### Pembacaan Data Bersih Lintas Sektor
---
Memuat dataset makroekonomi, tarif sewa, densitas kompetitor OSM, dan sensus SiBakul untuk membentuk matriks analisis.


In [ ]:
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'analisis' else Path.cwd()
CLEANED_DIR = BASE_DIR / 'data' / 'cleaned'
HASIL_DIR = BASE_DIR / 'outputs' / 'hasil'
OUTPUT_DIR = HASIL_DIR
df_kompetitor = pd.read_csv(CLEANED_DIR / 'kompetitor_per_wilayah.csv')
df_ekonomi = pd.read_csv(CLEANED_DIR / 'dataset2_bps_makro_cleaned.csv')
df_sewa = pd.read_csv(CLEANED_DIR / 'biaya_operasional.csv')
df_sibakul = pd.read_csv(CLEANED_DIR / 'sibakul_umkm_bersih.csv')


# 02 Pembentukan Matriks 25 Kombinasi Wilayah dan Sektor
---
## Cross Join Entitas
---
### Pemetaan 5 Kabupaten Kota dan 5 Sektor UMKM
---
Menggabungkan seluruh kombinasi wilayah dan sektor usaha untuk evaluasi kelayakan multi-kriteria.


In [ ]:
wilayah_list = ['sleman', 'bantul', 'kota_yogyakarta', 'kulon_progo', 'gunungkidul']
sektor_list = ['cafe', 'restaurant', 'convenience', 'laundry', 'grocery']
rows = []
for w in wilayah_list:
    for s in sektor_list:
        rows.append({'wilayah': w, 'sektor': s})
df_matrix = pd.DataFrame(rows)


In [ ]:
komp_counts = df_kompetitor.groupby(['wilayah', 'sektor']).size().reset_index(name='jumlah_kompetitor')
df_matrix = df_matrix.merge(komp_counts, on=['wilayah', 'sektor'], how='left').fillna({'jumlah_kompetitor': 0})
df_matrix = df_matrix.merge(df_ekonomi[['wilayah', 'pdrb_per_kapita_ribu_rp', 'pengeluaran_per_kapita_rp', 'kepadatan_jiwa_km2']], on='wilayah')
df_matrix = df_matrix.merge(df_sewa[['wilayah', 'tarif_sewa_per_m2_tahun']], on='wilayah')
df_matrix = df_matrix.merge(df_sibakul[['wilayah', 'densitas_per_1000']], on='wilayah')
df_matrix['rasio_kompetitor_per_1000'] = (df_matrix['jumlah_kompetitor'] / (df_matrix['kepadatan_jiwa_km2'] * 0.1)).round(2)
df_matrix.head()


# 03 Normalisasi Fitur Min-Max Scaling
---
## Transformasi Skala Bebas Satuan
---
### Penyeragaman Seluruh Kriteria ke Rentang Nol hingga Satu
---
Menerapkan Min-Max scaling agar perbedaan dimensi nominal rupiah dan rasio tidak membiaskan bobot kriteria.


In [ ]:
def min_max(col):
    diff = col.max() - col.min()
    return (col - col.min()) / diff if diff != 0 else 0.5
df_matrix['z_pdrb'] = min_max(df_matrix['pdrb_per_kapita_ribu_rp'])
df_matrix['z_pengeluaran'] = min_max(df_matrix['pengeluaran_per_kapita_rp'])
df_matrix['z_sibakul'] = min_max(df_matrix['densitas_per_1000'])
df_matrix['z_kompetitor'] = min_max(df_matrix['rasio_kompetitor_per_1000'])
df_matrix['z_sewa'] = min_max(df_matrix['tarif_sewa_per_m2_tahun'])
df_matrix[['wilayah', 'sektor', 'z_pdrb', 'z_pengeluaran', 'z_kompetitor', 'z_sewa']].head()


# 04 Formulasi Weighted Linear Combination
---
## Pembobotan Kriteria
---
### Kalkulasi Skor Komposit Kelayakan Nol hingga Seratus
---
Menghitung skor kelayakan menggunakan bobot benefit daya beli dan penalti cost sewa serta kompetitor.


In [ ]:
w_pdrb = 0.25
w_pengeluaran = 0.25
w_sibakul = 0.15
w_kompetitor = 0.20
w_sewa = 0.15
base_offset = 0.35
df_matrix['skor_raw'] = 100 * (
    w_pdrb * df_matrix['z_pdrb'] +
    w_pengeluaran * df_matrix['z_pengeluaran'] +
    w_sibakul * df_matrix['z_sibakul'] -
    w_kompetitor * df_matrix['z_kompetitor'] -
    w_sewa * df_matrix['z_sewa'] +
    base_offset
)
df_matrix['skor_kelayakan'] = df_matrix['skor_raw'].clip(0, 100).round(1)


# 05 Klasifikasi Ambang Batas Kelayakan
---
## Rule-based Categorization
---
### Penentuan Kategori Layak Cukup Layak dan Kurang Layak
---
Mengelompokkan 25 kombinasi sektor ke dalam kategori operasional bisnis.


In [ ]:
def label_kategori(skor):
    if skor >= 70:
        return 'Layak'
    elif skor >= 50:
        return 'Cukup Layak'
    else:
        return 'Kurang Layak'
df_matrix['kategori'] = df_matrix['skor_kelayakan'].apply(label_kategori)
df_matrix['kategori'].value_counts()


# 06 Visualisasi dan Ekspor Data
---
## Penyimpanan Hasil Skoring
---
### Ekspor Tabel Hasil Skoring Sektor ke CSV
---
Menyimpan hasil akhir pemodelan Pilar 1 untuk dibaca oleh mesin inferensi.


In [ ]:
cols_export = ['wilayah', 'sektor', 'skor_kelayakan', 'kategori', 'rasio_kompetitor_per_1000', 'tarif_sewa_per_m2_tahun']
df_hasil = df_matrix[cols_export].sort_values(by='skor_kelayakan', ascending=False)
output_p1 = OUTPUT_DIR / 'hasil_skoring_sektor.csv'
df_hasil.to_csv(output_p1, index=False)
print('Hasil skoring Pilar 1 berhasil disimpan di:', output_p1)
df_hasil.head(10)
